# Notebook 4 — A model trained for the task, against a model told to do the task

Notebook 1 showed that choosing the right model size, once, saves more energy than anything
decided at run time. This notebook tests the other half of the same idea: **does training a
small model for one task let it replace a much larger general one?**

That is the central claim made for small language models in organizations, and it is testable
on this machine because Qwen publishes a code-specialised model at exactly the sizes of the
general ladder used in notebook 1.

**Seven arms, three matched pairs.**

| Pair | Specialised | General |
|---|---|---|
| 0.5B | `qwen2.5-coder:0.5b` | `qwen2.5:0.5b` |
| 1.5B | `qwen2.5-coder:1.5b` | `qwen2.5:1.5b` |
| 3B | `qwen2.5-coder:3b` | `qwen2.5:3b` |
| — | — | `qwen2.5:7b` — the model an organization reaches for by default |

Same family, same tokenizer, same Q4_K_M quantization, same parameter count inside each pair.
**The only difference is what the model was trained on.** That is as close to a controlled
experiment as a laptop allows, and it is the design notebook 1 already uses for size.

**Every arm receives exactly the same prompt.** Unlike an extraction model, a code model is an
instruction-following model with the same chat template as its general twin, so there is no
format to adapt and nothing to argue about: one prompt, seven models.

**The grading is deterministic and costs nothing.** Each task asks for one Python function; the
function is run against a handful of assertions in a separate process. No judge, no rubric, no
energy spent on evaluation — which removes the largest source of doubt in notebook 1.

The instructions are in English. Code generation is an English-instruction task in practice, and
both arms were trained that way; writing them in French would mix language in with the thing
being measured.

Writes `outputs/coder_runs.csv`. Eight tasks × seven arms × three repetitions = 168 calls,
roughly 55 minutes.

## Before running

```
ollama pull qwen2.5-coder:0.5b
ollama pull qwen2.5-coder:1.5b
ollama pull qwen2.5-coder:3b
```

The four general models are already on the machine from notebook 1.

## A note on running generated code

Grading means executing what the model wrote. Each candidate runs in a separate Python process
with a ten-second limit and no standard input, so an infinite loop ends the call rather than the
notebook. The tasks are small pure functions over strings and numbers; nothing asks the model to
touch the filesystem or the network. This is how code benchmarks are graded, but it is worth
knowing that it is what the notebook does.

In [ ]:
%pip install -q requests pandas numpy matplotlib codecarbon psutil

In [ ]:
import re
import subprocess
import sys
import random

import numpy as np
import pandas as pd

from core import ask, save_table, SEED

ARMS = {
    "general 0.5B": {"model": "qwen2.5:0.5b",       "kind": "general",     "parameters_b": 0.5},
    "coder 0.5B":   {"model": "qwen2.5-coder:0.5b", "kind": "specialised", "parameters_b": 0.5},
    "general 1.5B": {"model": "qwen2.5:1.5b",       "kind": "general",     "parameters_b": 1.5},
    "coder 1.5B":   {"model": "qwen2.5-coder:1.5b", "kind": "specialised", "parameters_b": 1.5},
    "general 3B":   {"model": "qwen2.5:3b",         "kind": "general",     "parameters_b": 3.0},
    "coder 3B":     {"model": "qwen2.5-coder:3b",   "kind": "specialised", "parameters_b": 3.0},
    "general 7B":   {"model": "qwen2.5:7b",         "kind": "general",     "parameters_b": 7.0},
}

PAIRS = [("general 0.5B", "coder 0.5B"),
         ("general 1.5B", "coder 1.5B"),
         ("general 3B",   "coder 3B")]

REFERENCE = "general 7B"      # what an organization uses when it does not think about it
MAX_TOKENS = 400              # a function needs room; the ceiling is checked after the run

print(" / ".join(ARMS))
print(f"\n{len(PAIRS)} matched pairs, reference arm: {REFERENCE}")

## 1. Is Ollama up, and are the seven models installed?

Everything below talks to a server running on this machine. If that server is not started, every
call fails with the same connection error and the notebook prints seven identical stack traces,
which says nothing useful. This cell asks the question once and answers it in one line.

In [ ]:
import requests

from core import OLLAMA_URL


def preflight():
    """True when the server answers and every arm's model is installed."""
    try:
        catalogue = requests.get(f"{OLLAMA_URL}/api/tags", timeout=10).json()
    except requests.exceptions.RequestException:
        print(f"Ollama is not answering on {OLLAMA_URL}.\n")
        print("Nothing below can run until the server is started:")
        print("  Windows  open the Ollama application from the Start menu;")
        print("           it then sits in the notification area next to the clock")
        print("  any OS   or run  ollama serve  in a terminal and leave that window open\n")
        print("Check it with  ollama list  , then run this cell again.")
        return False

    installed = sorted(model.get("name", "") for model in catalogue.get("models", []))
    missing = [arm["model"] for arm in ARMS.values()
               if arm["model"] not in installed
               and f"{arm['model']}:latest" not in installed]

    print(f"Ollama is up, with {len(installed)} models installed.")
    if missing:
        print("\nMissing — pull each one, then run this cell again:")
        for model in missing:
            print(f"  ollama pull {model}")
        print("\ninstalled right now: " + (", ".join(installed) or "none"))
        return False

    print(f"All {len(ARMS)} arms are installed. Ready to run.")
    return True


READY = preflight()

## 2. Reading a model's answer, and running it

Two helpers. `extract_code` pulls the function out of the answer, because models wrap code in a
fenced block and often add a sentence around it. `run_tests` executes the candidate against the
task's assertions in a separate process and says why it failed when it did.

Storing the reason matters. A silent zero is unreadable — that is what made the previous version
of this notebook useless — so every failure records whether the code was missing, raised, timed
out, or simply returned the wrong answer.

In [ ]:
def extract_code(answer):
    """The Python the model meant to write, taken out of its fenced block if it used one."""
    text = str(answer)
    blocks = re.findall(r"```(?:python|py)?[ \t]*\n(.*?)(?:```|\Z)", text, re.S)
    for block in blocks:
        if "def " in block:
            return block
    if blocks:
        return blocks[0]
    return text if "def " in text else ""


def run_tests(candidate, tests, timeout=10):
    """Run the candidate against the task's assertions. Returns (correct, why)."""
    if not candidate.strip():
        return 0, "no code in the answer"
    program = candidate + "\n\n" + tests + "\nprint('ALL_TESTS_PASSED')"
    try:
        finished = subprocess.run([sys.executable, "-c", program],
                                  capture_output=True, text=True, timeout=timeout,
                                  stdin=subprocess.DEVNULL)
    except subprocess.TimeoutExpired:
        return 0, f"did not finish within {timeout} s"
    if "ALL_TESTS_PASSED" in finished.stdout:
        return 1, "pass"
    last_line = (finished.stderr.strip().splitlines() or ["no output"])[-1]
    return 0, last_line[:90]


print(extract_code("Here you go:\n```python\ndef add(a, b):\n    return a + b\n```\nHope it helps."))
print(run_tests("def add(a, b):\n    return a + b", "assert add(2, 3) == 5"))
print(run_tests("def add(a, b):\n    return a - b", "assert add(2, 3) == 5"))

## 3. Check every model is there and answers with code

A missing model does not raise an error with Ollama — it returns an empty answer, and the run
would silently produce a table of zeros. This cell asks each arm for a two-line function and
runs it. Thirty seconds, and a problem is visible before the fifty-five minutes start.

In [ ]:
PROBE_SPEC = ("Write a Python function add(a, b) that returns the sum of a and b.")
PROBE_TESTS = "assert add(2, 3) == 5\nassert add(-1, 1) == 0"

for name, arm in ARMS.items():
    try:
        probe = ask(arm["model"], PROBE_SPEC, max_new_tokens=120)
        answer = probe["answer"].strip()
        if not answer:
            verdict = "EMPTY — model probably not pulled"
        else:
            passed, why = run_tests(extract_code(answer), PROBE_TESTS)
            verdict = "ok" if passed else f"PROBLEM — {why}"
        print(f"{name:14} {arm['model']:22} {verdict}")
    except Exception as error:
        print(f"{name:14} {arm['model']:22} CALL FAILED: {type(error).__name__}: {error}")

## 4. Eight tasks

Each task carries a **specification** in plain English, the **assertions** that grade it, and a
**reference solution** used only to prove the assertions are fair before any model is called.

The difficulty is graded on purpose. `easy` is a one-liner; `medium` needs a data structure or a
known algorithm; `hard` needs the edge cases to be handled — an empty input, a bracket closing
the wrong kind, a run of length one. A test everyone passes separates nothing, and neither does
one everybody fails.

In [ ]:
TASKS = [
    {
        "id": "C1_count_vowels",
        "difficulty": "easy",
        "name": "count_vowels",
        "spec":
            "Write a Python function count_vowels(text) that returns the number of vowels in text.\n"
            "Vowels are a, e, i, o and u, in both lowercase and uppercase. Return an integer.\n"
        ,
        "tests":
            "assert count_vowels(\"hello\") == 2\n"
            "assert count_vowels(\"HELLO\") == 2\n"
            "assert count_vowels(\"xyz\") == 0\n"
            "assert count_vowels(\"\") == 0\n"
            "assert count_vowels(\"AeIoU\") == 5\n"
        ,
        "solution":
            "def count_vowels(text):\n"
            "    return sum(1 for c in text if c.lower() in \"aeiou\")\n"
        ,
    },
    {
        "id": "C2_second_largest",
        "difficulty": "easy",
        "name": "second_largest",
        "spec":
            "Write a Python function second_largest(numbers) that returns the second largest distinct\n"
            "value in the list numbers. If there are fewer than two distinct values, return None.\n"
        ,
        "tests":
            "assert second_largest([1, 2, 3]) == 2\n"
            "assert second_largest([5, 5, 4]) == 4\n"
            "assert second_largest([7]) is None\n"
            "assert second_largest([2, 2, 2]) is None\n"
            "assert second_largest([-1, -5, -3]) == -3\n"
        ,
        "solution":
            "def second_largest(numbers):\n"
            "    v = sorted(set(numbers), reverse=True)\n"
            "    return v[1] if len(v) > 1 else None\n"
        ,
    },
    {
        "id": "C3_group_by_length",
        "difficulty": "medium",
        "name": "group_by_length",
        "spec":
            "Write a Python function group_by_length(words) that returns a dictionary mapping each word\n"
            "length to the sorted list of words having that length.\n"
            "Example: group_by_length([\"bb\", \"a\", \"cc\"]) returns {2: [\"bb\", \"cc\"], 1: [\"a\"]}.\n"
        ,
        "tests":
            "assert group_by_length([\"bb\", \"a\", \"cc\"]) == {2: [\"bb\", \"cc\"], 1: [\"a\"]}\n"
            "assert group_by_length([]) == {}\n"
            "assert group_by_length([\"dog\", \"cat\", \"ox\"]) == {3: [\"cat\", \"dog\"], 2: [\"ox\"]}\n"
        ,
        "solution":
            "def group_by_length(words):\n"
            "    out = {}\n"
            "    for w in words:\n"
            "        out.setdefault(len(w), []).append(w)\n"
            "    return {k: sorted(v) for k, v in out.items()}\n"
        ,
    },
    {
        "id": "C4_roman_to_int",
        "difficulty": "medium",
        "name": "roman_to_int",
        "spec":
            "Write a Python function roman_to_int(roman) that converts a valid uppercase Roman numeral\n"
            "string into an integer. The subtractive pairs IV, IX, XL, XC, CD and CM must be handled.\n"
        ,
        "tests":
            "assert roman_to_int(\"III\") == 3\n"
            "assert roman_to_int(\"IV\") == 4\n"
            "assert roman_to_int(\"IX\") == 9\n"
            "assert roman_to_int(\"LVIII\") == 58\n"
            "assert roman_to_int(\"MCMXCIV\") == 1994\n"
        ,
        "solution":
            "def roman_to_int(roman):\n"
            "    v = {\"I\":1,\"V\":5,\"X\":10,\"L\":50,\"C\":100,\"D\":500,\"M\":1000}\n"
            "    total = 0\n"
            "    for i, c in enumerate(roman):\n"
            "        if i + 1 < len(roman) and v[c] < v[roman[i+1]]:\n"
            "            total -= v[c]\n"
            "        else:\n"
            "            total += v[c]\n"
            "    return total\n"
        ,
    },
    {
        "id": "C5_merge_intervals",
        "difficulty": "medium",
        "name": "merge_intervals",
        "spec":
            "Write a Python function merge_intervals(intervals) that takes a list of [start, end] pairs\n"
            "and returns the list of merged, non-overlapping intervals sorted by start.\n"
            "Intervals that merely touch, such as [1, 3] and [3, 5], must be merged into one.\n"
        ,
        "tests":
            "assert merge_intervals([[1,3],[2,6],[8,10]]) == [[1,6],[8,10]]\n"
            "assert merge_intervals([[1,4],[4,5]]) == [[1,5]]\n"
            "assert merge_intervals([]) == []\n"
            "assert merge_intervals([[5,6],[1,2]]) == [[1,2],[5,6]]\n"
        ,
        "solution":
            "def merge_intervals(intervals):\n"
            "    out = []\n"
            "    for s, e in sorted(intervals):\n"
            "        if out and s <= out[-1][1]:\n"
            "            out[-1][1] = max(out[-1][1], e)\n"
            "        else:\n"
            "            out.append([s, e])\n"
            "    return out\n"
        ,
    },
    {
        "id": "C6_is_balanced",
        "difficulty": "hard",
        "name": "is_balanced",
        "spec":
            "Write a Python function is_balanced(text) that returns True if the brackets in text are\n"
            "balanced and correctly nested, and False otherwise. The brackets are (), [] and {}.\n"
            "Every other character is ignored. An empty string is balanced.\n"
        ,
        "tests":
            "assert is_balanced(\"(a[b]{c})\") is True\n"
            "assert is_balanced(\"([)]\") is False\n"
            "assert is_balanced(\"(\") is False\n"
            "assert is_balanced(\"\") is True\n"
            "assert is_balanced(\"no brackets here\") is True\n"
        ,
        "solution":
            "def is_balanced(text):\n"
            "    pairs = {\")\":\"(\", \"]\":\"[\", \"}\":\"{\"}\n"
            "    stack = []\n"
            "    for c in text:\n"
            "        if c in \"([{\":\n"
            "            stack.append(c)\n"
            "        elif c in pairs:\n"
            "            if not stack or stack.pop() != pairs[c]:\n"
            "                return False\n"
            "    return not stack\n"
        ,
    },
    {
        "id": "C7_longest_common_prefix",
        "difficulty": "hard",
        "name": "longest_common_prefix",
        "spec":
            "Write a Python function longest_common_prefix(words) that returns the longest string which\n"
            "is a prefix of every word in the list. Return \"\" if the list is empty or the words share\n"
            "no common prefix.\n"
        ,
        "tests":
            "assert longest_common_prefix([\"flower\",\"flow\",\"flight\"]) == \"fl\"\n"
            "assert longest_common_prefix([\"dog\",\"racecar\"]) == \"\"\n"
            "assert longest_common_prefix([]) == \"\"\n"
            "assert longest_common_prefix([\"abc\"]) == \"abc\"\n"
            "assert longest_common_prefix([\"same\",\"same\"]) == \"same\"\n"
        ,
        "solution":
            "def longest_common_prefix(words):\n"
            "    if not words:\n"
            "        return \"\"\n"
            "    prefix = words[0]\n"
            "    for w in words[1:]:\n"
            "        while not w.startswith(prefix):\n"
            "            prefix = prefix[:-1]\n"
            "            if not prefix:\n"
            "                return \"\"\n"
            "    return prefix\n"
        ,
    },
    {
        "id": "C8_compress",
        "difficulty": "hard",
        "name": "compress",
        "spec":
            "Write a Python function compress(text) that performs run-length encoding: every run of the\n"
            "same character is replaced by that character followed by the length of the run, except a\n"
            "run of length 1, which is left as the single character.\n"
            "Example: compress(\"aaabbc\") returns \"a3b2c\".\n"
        ,
        "tests":
            "assert compress(\"aaabbc\") == \"a3b2c\"\n"
            "assert compress(\"abc\") == \"abc\"\n"
            "assert compress(\"\") == \"\"\n"
            "assert compress(\"aaaa\") == \"a4\"\n"
            "assert compress(\"aabbaa\") == \"a2b2a2\"\n"
        ,
        "solution":
            "def compress(text):\n"
            "    if not text:\n"
            "        return \"\"\n"
            "    out = []\n"
            "    run = 1\n"
            "    for i in range(1, len(text) + 1):\n"
            "        if i < len(text) and text[i] == text[i-1]:\n"
            "            run += 1\n"
            "        else:\n"
            "            out.append(text[i-1] if run == 1 else text[i-1] + str(run))\n"
            "            run = 1\n"
            "    return \"\".join(out)\n"
        ,
    },
]

counts = pd.Series([t["difficulty"] for t in TASKS]).value_counts()
print(f"{len(TASKS)} tasks  ({', '.join(f'{n} {d}' for d, n in counts.items())})")
print(f"{len(TASKS) * len(ARMS) * 3} calls in total")

### The assertions accept a correct function and reject an empty one

Before any model is called, every task is graded twice: once with its reference solution, which
must pass, and once with a function that returns `None`, which must fail. A grader that rejects
correct work turns a working model into a zero, which is exactly how the previous version of
this notebook came to report 0 % for a model that was answering perfectly.

In [ ]:
problems = []
for task in TASKS:
    passed, why = run_tests(task["solution"], task["tests"])
    if not passed:
        problems.append(f"{task['id']}: the reference solution FAILS — {why}")
    empty = f"def {task['name']}(*args, **kwargs):\n    return None"
    passed_empty, _ = run_tests(empty, task["tests"])
    if passed_empty:
        problems.append(f"{task['id']}: a function returning None PASSES — the test is too weak")
    no_code, _ = run_tests("", task["tests"])
    if no_code:
        problems.append(f"{task['id']}: an empty answer PASSES")

print("\n".join(problems) if problems
      else f"All {len(TASKS)} graders accept their reference solution and reject an empty one.")

## 5. The prompt

One prompt, used for all seven arms. The specialised models get no advantage in the wording and
no handicap either — which is the whole point of pairing a model with its own twin.

In [ ]:
def build_prompt(task):
    return ("Write a Python function that satisfies this specification.\n"
            "Reply with the function only, inside a single ```python code block.\n"
            "Do not add tests, examples or explanations.\n\n"
            + task["spec"])


print(build_prompt(TASKS[5]))

## 6. Run

The model is unloaded before every call, and the order of the seven arms is shuffled within each
task with a fixed seed — the same two precautions as everywhere else. The CSV is written after
every call, so an interrupted run still leaves everything measured up to that point.

The tests run **after** the energy measurement has stopped, so no part of the grading is counted
as inference energy.

In [ ]:
if not READY:
    raise RuntimeError("Ollama is not ready — see section 1. Nothing was run.")

RUNS_FILE = "outputs/coder_runs.csv"
REPETITIONS = 3

schedule = []
for task_index, task in enumerate(TASKS):
    for repeat in range(REPETITIONS):
        order = list(ARMS)
        random.Random(SEED + task_index * 10 + repeat).shuffle(order)
        for arm_name in order:
            schedule.append((task, arm_name, repeat))

rows = []
for position, (task, arm_name, repeat) in enumerate(schedule, start=1):
    arm = ARMS[arm_name]
    result = ask(arm["model"], build_prompt(task), max_new_tokens=MAX_TOKENS)
    candidate = extract_code(result["answer"])
    correct, why = run_tests(candidate, task["tests"])

    rows.append({
        "task": task["id"], "difficulty": task["difficulty"], "arm": arm_name,
        "model": arm["model"], "kind": arm["kind"],
        "parameters_b": arm["parameters_b"], "repeat": repeat,
        "energy_wh": result["energy_wh"], "duration_s": result["duration_s"],
        "load_s": result["load_s"], "tokens_out": result["tokens_out"],
        "correct": correct, "why": why, "code": candidate[:400],
    })
    pd.DataFrame(rows).to_csv(RUNS_FILE, index=False, encoding="utf-8-sig")

    print(f"[{position:3}/{len(schedule)}] {task['id']:26} {arm_name:13} "
          f"{result['energy_wh']:7.4f} Wh  {result['duration_s']:5.1f} s  "
          f"{result['tokens_out']:3} tok  {'pass' if correct else 'FAIL: ' + why[:38]}")

runs = pd.read_csv(RUNS_FILE)
print(f"\n{len(runs)} calls in {RUNS_FILE}")

## 7. Check the measurement before reading it

Three things have to hold before the table below means anything.

**Every arm must show a real model-loading time.** An arm near zero found its model already in
memory and did not pay what the others paid.

**No arm may sit at the token ceiling.** An arm whose answers are all exactly the maximum length
did not finish writing its function; its energy then measures the budget rather than the model,
and its code is truncated mid-line.

**Failures must be real failures.** If an arm's errors are mostly `no code in the answer`, it is
not failing at programming — it is failing to use a fenced block, which is a parsing problem and
must be fixed before the accuracy column is read.

In [ ]:
report = runs.groupby("arm").agg(load_s=("load_s", "mean"),
                                 tokens_out=("tokens_out", "mean"),
                                 at_ceiling=("tokens_out", lambda s: (s >= MAX_TOKENS).mean()))
report["at_ceiling %"] = 100 * report.at_ceiling
print(report[["load_s", "tokens_out", "at_ceiling %"]].reindex(list(ARMS)).round(2).to_string())

print("\nwhy the failures happened")
failures = runs[runs.correct == 0]
if len(failures):
    print(failures.why.str.slice(0, 44).value_counts().head(12).to_string())
else:
    print("  no failures")

no_code = failures.why.eq("no code in the answer").mean() if len(failures) else 0
warnings = []
if report.load_s.min() < 0.2:
    warnings.append("at least one arm never loaded its model")
if report.at_ceiling.max() > 0.1:
    warnings.append("at least one arm is hitting the token ceiling")
if no_code > 0.3:
    warnings.append("most failures are unparsed answers, not wrong code")

print("\nPROBLEM: " + "; ".join(warnings) + ". Do not use these figures."
      if warnings else "\nFine: every arm loaded its model, finished its answers, "
                       "and its failures are real failures.")

## 8. The result

In [ ]:
summary = runs.groupby("arm").agg(
    calls=("correct", "size"),
    parameters_b=("parameters_b", "first"),
    kind=("kind", "first"),
    energy_wh=("energy_wh", "mean"),
    latency_s=("duration_s", "mean"),
    accuracy=("correct", "mean"))
summary["accuracy %"] = 100 * summary.accuracy
# what one correct answer costs: the figure an organization actually pays
summary["wh_per_correct"] = np.where(summary.accuracy > 0,
                                     summary.energy_wh / summary.accuracy.replace(0, np.nan),
                                     np.nan)
summary = summary.reindex(list(ARMS))
save_table(summary.reset_index().round(5), "coder_summary")
print(summary[["calls", "parameters_b", "energy_wh", "latency_s",
               "accuracy %", "wh_per_correct"]].round(4).to_string())

print("\nInside each matched pair — same size, same quantization, different training")
for general, coder in PAIRS:
    g, c = summary.loc[general], summary.loc[coder]
    print(f"  {general:13} -> {coder:11} "
          f"accuracy {g['accuracy %']:5.1f} % -> {c['accuracy %']:5.1f} %   "
          f"({c['accuracy %'] - g['accuracy %']:+5.1f} points)   "
          f"energy {100 * (c.energy_wh - g.energy_wh) / g.energy_wh:+6.1f} %")

print(f"\nAgainst {REFERENCE}, the model an organization reaches for by default")
reference = summary.loc[REFERENCE]
for arm in summary.index:
    if arm == REFERENCE:
        continue
    row = summary.loc[arm]
    verdict = "matches or beats it" if row.accuracy >= reference.accuracy else ""
    print(f"  {arm:13} accuracy {row['accuracy %']:5.1f} % vs {reference['accuracy %']:5.1f} %   "
          f"energy {100 * (row.energy_wh - reference.energy_wh) / reference.energy_wh:+6.1f} %   "
          f"{verdict}")

print("\naccuracy by difficulty (%)")
print((100 * runs.pivot_table(index="difficulty", columns="arm", values="correct",
                             aggfunc="mean").reindex(columns=list(ARMS))
       .reindex(["easy", "medium", "hard"])).round(0).to_string())

print("\naccuracy by task (%)")
print((100 * runs.pivot_table(index="task", columns="arm", values="correct", aggfunc="mean")
       .reindex(columns=list(ARMS))).round(0).to_string())

## 9. Figure

Energy against accuracy. Each matched pair is joined by a line, so the effect of training — and
only of training — is the length and direction of that line. The dashed line is the accuracy of
the default 7B model: any point above it is a model that does the job for less.

In [ ]:
from core import new_figure, save_figure

GENERAL, SPECIALISED, LINK = "#1a4e8a", "#e07b39", "#C8CDD4"
points = summary.copy()

figure, axes = new_figure(
    "A model trained for the task, against a model told to do the task",
    "Eight Python tasks graded by running the code. Each line joins two models of the same "
    "family, size and quantization; only the training differs.",
    "Tasks solved (%)", width=10.5, height=6.4)

for general, coder in PAIRS:
    g, c = points.loc[general], points.loc[coder]
    axes.plot([g.energy_wh, c.energy_wh], [g["accuracy %"], c["accuracy %"]],
              color=LINK, linewidth=1.4, zorder=2)

for arm in points.index:
    row = points.loc[arm]
    colour = SPECIALISED if row.kind == "specialised" else GENERAL
    axes.scatter(row.energy_wh, row["accuracy %"], s=250, color=colour, zorder=3)
    above = row.kind == "specialised"
    axes.annotate(f"{arm} — {row['accuracy %']:.0f} %",
                  xy=(row.energy_wh, row["accuracy %"]),
                  xytext=(0, 15 if above else -24), textcoords="offset points",
                  fontsize=10, color="#5A6472", ha="center")

reference_accuracy = points.loc[REFERENCE, "accuracy %"]
axes.axhline(reference_accuracy, color=GENERAL, linestyle="--", linewidth=1.1, zorder=1)
axes.text(points.energy_wh.max() * 0.015, reference_accuracy + 3.2,
          f"what the default {REFERENCE.split()[-1]} model solves", fontsize=9.5,
          color=GENERAL, ha="left", style="italic")

axes.set_xlabel("Energy per request (Wh)", fontsize=10, color="#5A6472")
axes.set_xlim(0, points.energy_wh.max() * 1.12)
axes.set_ylim(-8, 112)
axes.grid(axis="x", color="#E4E4E1", linewidth=0.8, zorder=0)
save_figure(figure, axes, "F8_specialised_models")

## 10. Output of this notebook

In `outputs/`: `coder_runs.csv`, `coder_summary.csv` and
`figures/F8_specialised_models.png`. Send me all three.

`coder_runs.csv` keeps the first 400 characters of every function written and the reason each
failure failed, so any number in the table above can be traced back to what a model actually
produced.